# 02d - Images in the style of Shalabi et al. (2026)

Feature file of the `obspy_spectrogram` representation, for the replication of Shalabi et al. on ESM25
(`wqu.preprocessing.obspy_spectrogram`): for each component, the ObsPy spectrogram (`Trace.spectrogram` with its
default settings: Hann window of 128 samples, 90% overlap, zero-padding to 1024 samples) of the recorded samples at
the original sampling rate of the record, power in decibels scaled to [0, 1], reduced to 64 x 64. Every step is an
option of `preprocessing.obspy_spectrogram` in `configs/default.yaml`, to change one element at a time.

- Metadata and traces are read from the full SeisBench datasets, whatever `data.source` and `waveforms.source` say.
- Output: `data/processed/seisbench/obspy/<settings>/features.h5` with `records.csv` (records with at least one
  component with data) and `excluded_records.csv` (records without data: the same records excluded by the time-masked
  networks, so that the runs use the same records).
- The run is **resumable**: if it stops, run the notebook again and it continues from the records not yet processed.
  If the file is already complete, the computation is skipped.

To train on these images (notebook 05): `training.representation: obspy_spectrogram`, `training.standardization:
none`, `training.class_weighting: none`, `training.batch_size: 128`, `model.architecture: shalabi_inception`,
`model.time_masking: false`, and the split of your choice (`split.grouping`).

## 0. Setup

In [ ]:
from dataclasses import replace
from pathlib import Path
import shutil

import numpy as np
import pandas as pd
import torch

from wqu.config import OBSPY_REPRESENTATION, load_config
from wqu.data.exploration import numeric_summary_by_label
from wqu.data.loading import CV_COMPONENTS, load_esm25_metadata
from wqu.data.waveforms import iter_waveform_batches, stored_sampling_rate_hz
from wqu.models.complexity import model_complexity
from wqu.models.cred import build_model
from wqu.models.uncertainty import mc_dropout_probabilities
from wqu.preprocessing.obspy_features import (
    DURATION_COLUMN_FORMAT,
    HAS_DATA_COLUMN_FORMAT,
    SAMPLING_RATE_COLUMN,
    excluded_records_path,
    obspy_feature_path,
    store_obspy_features,
)
from wqu.preprocessing.pipeline import FEATURE_INDEX_COLUMN, read_feature_records, read_features
from wqu.visualization import apply_plot_style, plot_obspy_images, save_figure

CONFIG_PATH = Path("..") / "configs" / "default.yaml"
FULL_SOURCE = "seisbench"
BYTES_PER_VALUE = 2
N_EXAMPLES_PER_CLASS = 4
CHECK_BATCH_PER_CLASS = 16

# Metadata and traces from the SeisBench datasets, whatever data.source and waveforms.source say.
config = load_config(CONFIG_PATH)
config = replace(
    config,
    data=replace(config.data, source=FULL_SOURCE),
    waveforms=replace(config.waveforms, source=FULL_SOURCE),
)
columns = config.columns
preprocessing = config.preprocessing
settings = preprocessing.obspy_spectrogram
if settings is None:
    raise ValueError("configs/default.yaml has no preprocessing.obspy_spectrogram section.")
label_names = {subset.label: subset.name for subset in config.data.subsets}
apply_plot_style()

_, metadata = load_esm25_metadata(config)
sampling_rate_hz = stored_sampling_rate_hz(metadata)
features_path = obspy_feature_path(preprocessing, FULL_SOURCE)
figures_dir = features_path.parent / "figures"
print(f"Records: {len(metadata):,}; stored traces at {sampling_rate_hz:g} Hz")
print(f"Settings: {settings}")
print(f"Output: {features_path}")

## 1. Disk space

In [ ]:
values_per_record = len(CV_COMPONENTS) * settings.n_frequencies * settings.n_times
required_bytes = len(metadata) * values_per_record * BYTES_PER_VALUE
features_path.parent.mkdir(parents=True, exist_ok=True)
free_bytes = shutil.disk_usage(features_path.parent).free
print(f"Feature file: {required_bytes / 1e9:.2f} GB; free space: {free_bytes / 1e9:.1f} GB")
if not features_path.exists() and free_bytes < required_bytes:
    raise RuntimeError("Not enough free disk space for the feature file.")

## 2. Images (resumable)

Only the traces are read and three short Fourier transforms are computed per record: about the time of notebook 02c.
The progress bar starts from the records already processed in a previous run.

In [ ]:
if features_path.exists():
    stored_records = read_feature_records(features_path, columns.text)
    print(f"Feature file already complete: {len(stored_records):,} records with data")
else:
    stored_records = store_obspy_features(
        metadata, lambda selected: iter_waveform_batches(selected, config), sampling_rate_hz, preprocessing,
        config.waveforms.zero_tolerance, features_path,
    )
    print(f"Stored {len(stored_records):,} records with data in {features_path}")
excluded_records = pd.read_csv(excluded_records_path(features_path), index_col=0)
print(f"Records without data (left out of records.csv): {len(excluded_records):,}")

## 3. Checks

Per class: records with data and without, share of records with each component without data, sampling rate of the
images and duration of the segments.

In [ ]:
has_data_columns = [HAS_DATA_COLUMN_FORMAT.format(component) for component in CV_COMPONENTS]
counts = pd.DataFrame({
    "with data": stored_records[columns.label].value_counts(),
    "without data": excluded_records[columns.label].value_counts(),
}).fillna(0).astype(int).rename(index=label_names)
missing_components = (~stored_records[has_data_columns].astype(bool)).groupby(stored_records[columns.label]).mean()
display(counts)
missing_components.rename(index=label_names).round(4)

In [ ]:
rates = (stored_records.groupby(columns.label)[SAMPLING_RATE_COLUMN].value_counts(normalize=True)
         .unstack(level=0).rename(columns=label_names).fillna(0.0))
display(rates.round(4))
duration_columns = [DURATION_COLUMN_FORMAT.format(component) for component in CV_COMPONENTS]
numeric_summary_by_label(stored_records, duration_columns, columns.label).rename(
    index=label_names, level=columns.label).round(1)

## 4. Examples

Random records of each class, as the network sees them (grey levels from 0 to 1; frequency upwards, time to the
right). Saved in the `figures` folder next to the feature file.

In [ ]:
examples = stored_records.groupby(columns.label, group_keys=False).sample(
    n=N_EXAMPLES_PER_CLASS, random_state=config.split.random_seed
)
example_images, _ = read_features(features_path, OBSPY_REPRESENTATION, examples[FEATURE_INDEX_COLUMN].to_numpy())
titles = [f"{label_names[label]}, {rate:g} Hz" for label, rate in zip(examples[columns.label], examples[SAMPLING_RATE_COLUMN])]
figure = plot_obspy_images(example_images, titles, CV_COMPONENTS, f"{OBSPY_REPRESENTATION}: {settings.variant_name}")
save_figure(figure, figures_dir / "examples.png")
save_figure(figure, figures_dir / "examples.pdf")

## 5. Network check

The configured network (`model.architecture`; `shalabi_inception` for the replication) on a batch of stored images:
trainable parameters, FLOPs per record (Shalabi et al. report 0.75 million parameters), output shape and spread of the
MC dropout passes of the untrained network.

In [ ]:
batch_records = stored_records.groupby(columns.label, group_keys=False).sample(
    n=CHECK_BATCH_PER_CLASS, random_state=config.split.random_seed
)
batch_images, _ = read_features(features_path, OBSPY_REPRESENTATION, batch_records[FEATURE_INDEX_COLUMN].to_numpy())
batch = torch.from_numpy(np.ascontiguousarray(batch_images, dtype=np.float32))
model = build_model(config.model, len(CV_COMPONENTS), settings.n_frequencies)
complexity = model_complexity(model, batch)
with torch.no_grad():
    logits = model.eval()(batch)
stochastic = mc_dropout_probabilities(model, batch, config.model.mc_samples)
print(f"Architecture: {config.model.architecture}; input {tuple(batch.shape)}; output {tuple(logits.shape)}")
print(f"Trainable parameters: {complexity.parameters:,}; GFLOPs per record: {complexity.flops_per_record / 1e9:.4f}")
print(f"MC dropout: {config.model.mc_samples} passes, mean spread across passes {stochastic.std(dim=0).mean().item():.4f}")